In [3]:
## Colab Setup (skip if running locally)
# Run the cell only on Google Colab.

!git config --global user.email 'andrexnardo80@gmail.com'
!git config --global user.name 'flaviofrasca'

import os

!git clone https://github.com/flaviofrasca/MaskArchitectureAnomaly_CourseProject.git
os.chdir('/content/MaskArchitectureAnomaly_CourseProject/eomt')
print(os.getcwd())

from google.colab import drive
drive.mount('/content/drive')

!pip install -q \
    "lightning==2.5.1.post0" \
    "timm==1.0.15" \
    "transformers==4.56.1" \
    "torchmetrics==1.7.1" \
    "jsonargparse[signatures]==4.38" \
    "pycocotools==2.0.8" \
    "fvcore==0.1.5.post20221221" \
    "wandb==0.19.10" \
    "scipy==1.15.2" \
    "gitignore_parser==0.1.12"

fatal: destination path 'MaskArchitectureAnomaly_CourseProject' already exists and is not an empty directory.
/content/MaskArchitectureAnomaly_CourseProject/eomt
Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
%%javascript
function keepAlive() {
    console.log("Keep-alive: " + new Date().toLocaleTimeString());
    window.scrollBy(0, 1);
    window.scrollBy(0, -1);
    setTimeout(keepAlive, 60000);
}
keepAlive();

<IPython.core.display.Javascript object>

In [5]:
import os, glob, torch
import torch.nn.functional as F
os.environ["WANDB_MODE"] = "disabled"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
COCO_CKPT = BASE + "/checkpoints/coco/eomt_coco.bin"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"   # directory separata dal run in corso
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

def extract_weights(ckpt_path, out_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save(state_dict, out_path)
    print(f"Weights saved → {out_path}")
    return out_path

def find_last_checkpoint(phase_dir):
    ckpts = glob.glob(f"{phase_dir}/*/.ckpt", recursive=True)
    if not ckpts:
        raise FileNotFoundError(f"Nessun checkpoint in {phase_dir}")
    return sorted(ckpts, key=os.path.getmtime)[-1]

In [6]:
import os, glob, torch
import torch.nn.functional as F
os.environ["WANDB_MODE"] = "disabled"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
COCO_CKPT = BASE + "/checkpoints/coco/eomt_coco.bin"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"   # directory separata dal run in corso
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

def extract_weights(ckpt_path, out_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save(state_dict, out_path)
    print(f"Weights saved → {out_path}")
    return out_path

def find_last_checkpoint(phase_dir):
    # BUG RISOLTO: aggiunti i due asterischi e sistemata l'estensione
    ckpts = glob.glob(f"{phase_dir}/*/.ckpt", recursive=True)

    # Fallback di sicurezza: cerca anche direttamente nella cartella (senza sottocartelle)
    if not ckpts:
        ckpts = glob.glob(f"{phase_dir}/*.ckpt")

    if not ckpts:
        raise FileNotFoundError(f"🚨 Nessun checkpoint trovato in {phase_dir}")

    return sorted(ckpts, key=os.path.getmtime)[-1]

In [7]:
import torch
import torch.nn.functional as F

ckpt = torch.load(COCO_CKPT, map_location="cpu", weights_only=False)
state_dict = ckpt.get("state_dict", ckpt)

# Rimuove criterion.empty_weight (non è un parametro del modello)
state_dict.pop("criterion.empty_weight", None)

# Interpola pos_embed: [1, 1600, 768] (40x40) → [1, 4096, 768] (64x64)
pos_embed = state_dict["network.encoder.backbone.pos_embed"]
h_old, w_old, h_new, w_new = 40, 40, 64, 64
pos_embed_4d = pos_embed.reshape(1, h_old, w_old, 768).permute(0, 3, 1, 2).float()
pos_embed_interp = F.interpolate(pos_embed_4d, size=(h_new, w_new), mode='bicubic', align_corners=False)
state_dict["network.encoder.backbone.pos_embed"] = pos_embed_interp.permute(0, 2, 3, 1).reshape(1, h_new * w_new, 768)
print(f"pos_embed: {pos_embed.shape} → {state_dict['network.encoder.backbone.pos_embed'].shape}")

# Taglia q.weight: [200, 768] → [100, 768]
q_weight = state_dict["network.q.weight"]
state_dict["network.q.weight"] = q_weight[:100, :]
print(f"q.weight: {q_weight.shape} → {state_dict['network.q.weight'].shape}")

FILTERED_COCO_CKPT = "/tmp/coco_interpolated.bin"
torch.save(state_dict, FILTERED_COCO_CKPT)
print(f"Checkpoint pronto → {FILTERED_COCO_CKPT}")

pos_embed: torch.Size([1, 1600, 768]) → torch.Size([1, 4096, 768])
q.weight: torch.Size([200, 768]) → torch.Size([100, 768])
Checkpoint pronto → /tmp/coco_interpolated.bin


In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

phase_dir = f"{SAVE_DIR}/phase1_head_only"
os.makedirs(phase_dir, exist_ok=True)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {FILTERED_COCO_CKPT}"
    f" --model.init_args.load_ckpt_class_head False"
    f" --model.init_args.llrd 0.0"
    f" --model.init_args.lr_mult 0.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 3"
    f" --trainer.default_root_dir {phase_dir}"
    ' "--trainer.callbacks+={\"class_path\": \"lightning.pytorch.callbacks.ModelCheckpoint\", \"init_args\": {\"save_last\": true, \"every_n_epochs\": 1}}"'
    f" --compile_disabled"
)
!{cmd}


2026-05-25 14:58:29.958459: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 195 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

   | Name                     | Type                        | Params | Mode 
----------------------------------------------------------------------------------
0  | netwo

FileNotFoundError: Nessun checkpoint in /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only

In [ ]:
CHUNK1_LAST_CKPT = find_last_checkpoint(phase_dir)
print(f"Chunk 1 done → {CHUNK1_LAST_CKPT}")

NameError: name 'phase_dir' is not defined

In [ ]:
import os
import shutil

# La cartella locale dove il training ha effettivamente salvato
source_dir = "/content/MaskArchitectureAnomaly_CourseProject/eomt/eomt/e4lzmz78/checkpoints/"

# La cartella su Drive dove il tuo script si aspetta di trovarli
target_dir = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only"

# Crea la cartella di destinazione se per caso non esiste
os.makedirs(target_dir, exist_ok=True)

# Copia tutti i file .ckpt al sicuro su Drive
for file in os.listdir(source_dir):
    if file.endswith(".ckpt"):
        src = os.path.join(source_dir, file)
        dst = os.path.join(target_dir, file)
        print(f"Copiando {file} su Drive...")
        shutil.copy2(src, dst)

print("\n✅ Salvataggio completato con successo! Ora i file sono su Drive.")

Copiando last.ckpt su Drive...
Copiando epoch=2-step=2232.ckpt su Drive...

✅ Salvataggio completato con successo! Ora i file sono su Drive.


In [ ]:
!ls -lh /content/MaskArchitectureAnomaly_CourseProject/eomt/eomt/e4lzmz78/checkpoints/

ls: cannot access '/content/MaskArchitectureAnomaly_CourseProject/eomt/eomt/e4lzmz78/checkpoints/': No such file or directory


In [ ]:
print("Il file selezionato è:", find_last_checkpoint(phase_dir))

Il file selezionato è: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only/last.ckpt


# CELLA NUOVA DI PROVA

In [9]:
#GIUSTO
import os
import glob
import re

def find_best_epoch_checkpoint(phase_dir):
    """
    Cerca il checkpoint con l'epoca più alta, ignorando intenzionalmente 'last.ckpt'.
    Questo protegge dai riavvii che potrebbero sovrascrivere 'last.ckpt'.
    """
    # 1. Trova tutti i file .ckpt
    ckpts = glob.glob(f"{phase_dir}/*/.ckpt", recursive=True)
    if not ckpts:
        ckpts = glob.glob(f"{phase_dir}/*.ckpt")

    if not ckpts:
        raise FileNotFoundError(f"🚨 Nessun checkpoint trovato in {phase_dir}")

    # 2. Filtra via 'last.ckpt' (lo usiamo solo se è letteralmente l'unico file rimasto)
    epoch_ckpts = [c for c in ckpts if "last.ckpt" not in os.path.basename(c)]

    if not epoch_ckpts:
        print("⚠️ Trovato solo last.ckpt, uso questo per forza.")
        return ckpts[0]

    # 3. Funzione per estrarre (Epoca, Step) dal nome del file
    def extract_epoch_and_step(filepath):
        # Cerca "epoch=X"
        epoch_match = re.search(r'epoch=(\d+)', filepath)
        epoch = int(epoch_match.group(1)) if epoch_match else -1

        # Cerca "step=Y"
        step_match = re.search(r'step=(\d+)', filepath)
        step = int(step_match.group(1)) if step_match else -1

        # Ritorna una tupla (es. (2, 2232)), che Python sa ordinare perfettamente
        return (epoch, step)

    # 4. Ordina i file in base alla tupla (epoca, step) e prendi il più alto
    epoch_ckpts_ordinati = sorted(epoch_ckpts, key=extract_epoch_and_step)

    miglior_ckpt = epoch_ckpts_ordinati[-1]
    print(f"🎯 Selezionato il checkpoint esatto: {os.path.basename(miglior_ckpt)}")
    return miglior_ckpt

# --- Testiamo subito la funzione ---
phase1_dir = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only"
CHUNK1_LAST_CKPT = find_best_epoch_checkpoint(phase1_dir)

🎯 Selezionato il checkpoint esatto: epoch=2-step=2232.ckpt


In [13]:
#GIUSTO
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
phase_dir = f"{SAVE_DIR}/phase1_head_only"
os.makedirs(phase_dir, exist_ok=True)

CHUNK1_CKPT = f"{phase_dir}/epoch=2-step=2232.ckpt"

callback = (
    '{"class_path": "lightning.pytorch.callbacks.ModelCheckpoint", '
    '"init_args": {'
    f'"dirpath": "{phase_dir}", '   # <-- percorso Drive esplicito
    '"save_last": true, '
    '"every_n_epochs": 1}}'
)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {CHUNK1_CKPT}"
    f" --model.init_args.load_ckpt_class_head True"
    f" --model.init_args.llrd 0.0"
    f" --model.init_args.lr_mult 0.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 3"
    f' "--trainer.callbacks+={callback}"'
    f" --compile_disabled"
)
!{cmd}


2026-05-26 14:02:25.679867: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 197 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treesp

# CONTROLLO PESI CORRETTI

In [15]:
import glob, re, os

phase_dir = "/content/drive/MyDrive/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only"

ckpts = glob.glob(f"{phase_dir}/**/*.ckpt", recursive=True)
epoch_ckpts = [c for c in ckpts if "last.ckpt" not in os.path.basename(c)]

print("Tutti i checkpoint trovati:")
for c in ckpts:
    print(f"  {c}")

print("\nCheckpoint che verrebbe selezionato:")
def extract_epoch_and_step(filepath):
    epoch = int(m.group(1)) if (m := re.search(r'epoch=(\d+)', filepath)) else -1
    step  = int(m.group(1)) if (m := re.search(r'step=(\d+)',  filepath)) else -1
    return (epoch, step)

best = sorted(epoch_ckpts, key=extract_epoch_and_step)[-1]
print(f"  → {best}")

Tutti i checkpoint trovati:
  /content/drive/MyDrive/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only/epoch=2-step=2232.ckpt
  /content/drive/MyDrive/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only/last.ckpt
  /content/drive/MyDrive/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only/last-v1.ckpt
  /content/drive/MyDrive/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only/epoch=2-step=2232-v1.ckpt

Checkpoint che verrebbe selezionato:
  → /content/drive/MyDrive/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only/epoch=2-step=2232-v1.ckpt


In [14]:
#GIUSTO
import os, glob, re, torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
phase_dir = f"{SAVE_DIR}/phase1_head_only"

def find_best_epoch_checkpoint(phase_dir):
    ckpts = glob.glob(f"{phase_dir}/**/*.ckpt", recursive=True)
    if not ckpts:
        raise FileNotFoundError(f"Nessun checkpoint trovato in {phase_dir}")
    epoch_ckpts = [c for c in ckpts if "last.ckpt" not in os.path.basename(c)]
    if not epoch_ckpts:
        return ckpts[0]
    def extract_epoch_and_step(filepath):
        epoch = int(m.group(1)) if (m := re.search(r'epoch=(\d+)', filepath)) else -1
        step  = int(m.group(1)) if (m := re.search(r'step=(\d+)',  filepath)) else -1
        return (epoch, step)
    return sorted(epoch_ckpts, key=extract_epoch_and_step)[-1]

def extract_weights(ckpt_path, out_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save(state_dict, out_path)
    print(f"Weights saved → {out_path}")
    return out_path

CHUNK2_CKPT = find_best_epoch_checkpoint(phase_dir)
print(f"Riprendo da: {CHUNK2_CKPT}")

callback = (
    '{"class_path": "lightning.pytorch.callbacks.ModelCheckpoint", '
    '"init_args": {'
    f'"dirpath": "{phase_dir}", '   # <-- Drive esplicito
    '"save_last": true, '
    '"every_n_epochs": 1}}'
)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {CHUNK2_CKPT}"
    f" --model.init_args.load_ckpt_class_head True"
    f" --model.init_args.llrd 0.0"
    f" --model.init_args.lr_mult 0.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 4"
    f' "--trainer.callbacks+={callback}"'
    f" --compile_disabled"
)
!{cmd}

PHASE1_CKPT = extract_weights(
    find_best_epoch_checkpoint(phase_dir),
    f"{SAVE_DIR}/eomt_phase1_head_only.bin"
)
print(f"Phase 1 done → {PHASE1_CKPT}")

Riprendo da: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only/epoch=2-step=2232-v1.ckpt
2026-05-26 16:33:13.776360: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 197 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0

In [ ]:
#GIUSTO
import os, glob, re, torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
phase_dir = f"{SAVE_DIR}/phase1_head_only"

def find_best_epoch_checkpoint(phase_dir):
    ckpts = glob.glob(f"{phase_dir}/**/*.ckpt", recursive=True)
    if not ckpts:
        raise FileNotFoundError(f"Nessun checkpoint trovato in {phase_dir}")
    epoch_ckpts = [c for c in ckpts if "last.ckpt" not in os.path.basename(c)]
    if not epoch_ckpts:
        return ckpts[0]
    def extract_epoch_and_step(filepath):
        epoch = int(m.group(1)) if (m := re.search(r'epoch=(\d+)', filepath)) else -1
        step  = int(m.group(1)) if (m := re.search(r'step=(\d+)',  filepath)) else -1
        return (epoch, step)
    return sorted(epoch_ckpts, key=extract_epoch_and_step)[-1]

def extract_weights(ckpt_path, out_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save(state_dict, out_path)
    print(f"Weights saved → {out_path}")
    return out_path

# Trova il checkpoint salvato dal chunk precedente
CHUNK2_CKPT = find_best_epoch_checkpoint(phase_dir)
print(f"Riprendo da: {CHUNK2_CKPT}")

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {CHUNK2_CKPT}"     # solo pesi, no optimizer state
    f" --model.init_args.load_ckpt_class_head True"   # mantieni class head già trainata
    f" --model.init_args.llrd 0.0"
    f" --model.init_args.lr_mult 0.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 4"                         # 4 nuove epoche
    f" --trainer.default_root_dir {phase_dir}"
    ' "--trainer.callbacks+={\"class_path\": \"lightning.pytorch.callbacks.ModelCheckpoint\", \"init_args\": {\"save_last\": true, \"every_n_epochs\": 1}}"'
    f" --compile_disabled"
)
!{cmd}

PHASE1_CKPT = extract_weights(
    find_best_epoch_checkpoint(phase_dir),
    f"{SAVE_DIR}/eomt_phase1_head_only.bin"
)
print(f"Phase 1 done → {PHASE1_CKPT}")


In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

phase_dir = f"{SAVE_DIR}/phase1_head_only"
# Se la sessione è stata resettata:
# CHUNK2_LAST_CKPT = find_last_checkpoint(phase_dir)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path null"
    f" --model.init_args.load_ckpt_class_head False"
    f" --model.init_args.llrd 0.0"
    f" --model.init_args.lr_mult 0.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 10"
    f" --trainer.default_root_dir {phase_dir}"
    f" --trainer.ckpt_path {CHUNK2_LAST_CKPT}"
    ' "--trainer.callbacks+={\"class_path\": \"lightning.pytorch.callbacks.ModelCheckpoint\", \"init_args\": {\"save_last\": true, \"every_n_epochs\": 1}}"'
    f" --compile_disabled"
)
!{cmd}

PHASE1_CKPT = extract_weights(
    find_last_checkpoint(phase_dir),
    f"{SAVE_DIR}/eomt_phase1_head_only.bin"
)
print(f"Phase 1 done → {PHASE1_CKPT}")

#Altra parte

In [ ]:
import os, glob, torch
os.environ["WANDB_MODE"] = "disabled"

BASE = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
COCO_CKPT = BASE + "/checkpoints/coco/eomt_coco.bin"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

def extract_weights(ckpt_path, out_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save(state_dict, out_path)
    print(f"Weights saved → {out_path}")
    return out_path

def find_last_checkpoint(phase_dir):
    ckpts = glob.glob(f"{phase_dir}/**/*.ckpt", recursive=True)
    if not ckpts:
        raise FileNotFoundError(f"Nessun checkpoint in {phase_dir}")
    return sorted(ckpts, key=os.path.getmtime)[-1]

In [ ]:
phase_dir = f"{SAVE_DIR}/phase1_head_only"
os.makedirs(phase_dir, exist_ok=True)

!python main.py fit \
  --config {CONFIG} \
  --model.init_args.ckpt_path {COCO_CKPT} \
  --model.init_args.load_ckpt_class_head False \
  --model.init_args.llrd 0.0 \
  --model.init_args.lr_mult 0.0 \
  --model.init_args.attn_mask_annealing_enabled False \
  --data.init_args.path {DATA_PATH} \
  --data.init_args.batch_size 4 \
  --data.init_args.num_workers 2 \
  --trainer.max_epochs 10 \
  --trainer.default_root_dir {phase_dir} \
  "--trainer.callbacks+={\"class_path\": \"lightning.pytorch.callbacks.ModelCheckpoint\", \"init_args\": {\"save_last\": true, \"every_n_epochs\": 1}}" \
  --compile_disabled

PHASE1_CKPT = extract_weights(find_last_checkpoint(phase_dir), f"{phase_dir}/weights.bin")
print(f"Phase 1 done → {PHASE1_CKPT}")

In [ ]:
phase_dir = f"{SAVE_DIR}/phase2_unfreeze_last"
os.makedirs(phase_dir, exist_ok=True)

!python main.py fit \
  --config {CONFIG} \
  --model.init_args.ckpt_path {PHASE1_CKPT} \
  --model.init_args.load_ckpt_class_head True \
  --model.init_args.llrd 0.3 \
  --model.init_args.attn_mask_annealing_enabled False \
  --data.init_args.path {DATA_PATH} \
  --data.init_args.batch_size 4 \
  --data.init_args.num_workers 2 \
  --trainer.max_epochs 5 \
  --trainer.default_root_dir {phase_dir} \
  "--trainer.callbacks+={\"class_path\": \"lightning.pytorch.callbacks.ModelCheckpoint\", \"init_args\": {\"save_last\": true, \"every_n_epochs\": 1}}" \
  --compile_disabled

PHASE2_CKPT = extract_weights(find_last_checkpoint(phase_dir), f"{phase_dir}/weights.bin")
print(f"Phase 2 done → {PHASE2_CKPT}")

In [ ]:
phase_dir = f"{SAVE_DIR}/phase3_unfreeze_more"
os.makedirs(phase_dir, exist_ok=True)

!python main.py fit \
  --config {CONFIG} \
  --model.init_args.ckpt_path {PHASE2_CKPT} \
  --model.init_args.load_ckpt_class_head True \
  --model.init_args.llrd 0.6 \
  --model.init_args.attn_mask_annealing_enabled False \
  --data.init_args.path {DATA_PATH} \
  --data.init_args.batch_size 4 \
  --data.init_args.num_workers 2 \
  --trainer.max_epochs 5 \
  --trainer.default_root_dir {phase_dir} \
  "--trainer.callbacks+={\"class_path\": \"lightning.pytorch.callbacks.ModelCheckpoint\", \"init_args\": {\"save_last\": true, \"every_n_epochs\": 1}}" \
  --compile_disabled

PHASE3_CKPT = extract_weights(find_last_checkpoint(phase_dir), f"{phase_dir}/weights.bin")
print(f"Phase 3 done → {PHASE3_CKPT}")

In [ ]:
phase_dir = f"{SAVE_DIR}/phase4_full"
os.makedirs(phase_dir, exist_ok=True)

!python main.py fit \
  --config {CONFIG} \
  --model.init_args.ckpt_path {PHASE3_CKPT} \
  --model.init_args.load_ckpt_class_head True \
  --model.init_args.llrd 0.8 \
  --model.init_args.attn_mask_annealing_enabled False \
  --data.init_args.path {DATA_PATH} \
  --data.init_args.batch_size 4 \
  --data.init_args.num_workers 2 \
  --trainer.max_epochs 5 \
  --trainer.default_root_dir {phase_dir} \
  "--trainer.callbacks+={\"class_path\": \"lightning.pytorch.callbacks.ModelCheckpoint\", \"init_args\": {\"save_last\": true, \"every_n_epochs\": 1}}" \
  --compile_disabled

PHASE4_CKPT = extract_weights(find_last_checkpoint(phase_dir), f"{phase_dir}/weights.bin")
print(f"Phase 4 done → {PHASE4_CKPT}")